In [13]:
import os
os.environ["OPENAI_API_KEY"] = "<your OpenAI api key>"
from llmner import FewShotNer
from llmner.data import AnnotatedDocument, Annotation

# === Ollama config (free, local Large Language Model) ===
USE_OLLAMA = True
OLLAMA_MODEL = "ministral-3:14b"   # "ministral-3:14b" is a 14B-parameter model, good for NER tasks; see https://ollama.com/models for more options

if USE_OLLAMA:
    os.environ["OPENAI_API_KEY"] = "INNOVAFERTANIMUSMUTANTASDICEREFORMAS"     # placeholder  
    os.environ["OPENAI_API_BASE"] = "http://localhost:11434/v1"

In [ ]:
ENTITIES = {
    "person":       "A personal name, e.g. Cicero, Caesar, Catilina",
    "group":        "A people, tribe, army, or collective group, e.g. Romani, Belgae, Senatus",
    "geography":     "A geographic location: city, region, river, mountain, e.g. Roma, Gallia, Rhenus",
}

LABEL_MAP = {
    "person":       "PERS",
    "group":        "GRP",
    "geography":     "GEO",
}

EXAMPLES = [
    AnnotatedDocument(
        text="Arma virumque canō, Trōiae quī prīmus ab ōrīs Ītaliam",
        annotations={
            Annotation(start=20, end=25, label="geography"),
            Annotation(start=46, end=52, label="geography"),
        },
    ),
    AnnotatedDocument(
        text="inferretque deōs Latiō, genus unde Latīnum, Albānīque patrēs",
        annotations={
            Annotation(start=17, end=21, label="geography"),
            Annotation(start=35, end=41, label="group"),
            Annotation(start=44, end=52, label="group"),
        },
    ),
    AnnotatedDocument(
        text="Tu quoque litoribus nostris, Aeneia nutrix,",
        annotations={
            Annotation(start=29, end=34, label="person"),
        },
    ),
    AnnotatedDocument(
        text="proxima Circaeae raduntur litora terrae,",
        annotations={
            Annotation(start=8, end=15, label="person")
        },
    ),
]

In [15]:
model = FewShotNer(model=OLLAMA_MODEL)          # pass model="gpt-4.1-nano" etc. if using API
model.contextualize(entities=ENTITIES, examples=EXAMPLES)

In [16]:
# === Helpers to convert output into proper labels ===
def spans_to_bio(text: str, annotations) -> list[tuple[str, str]]:
    """
    Tokenise `text` on whitespace and map character-level Annotation spans
    onto BIO labels.  Returns [(token, tag), ...].

    Annotation objects expose:  .start  .end  .label
    (end is exclusive, matching Python slice convention)
    """
    # Build a char → (label, is_begin) lookup from sorted, non-overlapping spans
    char_label: dict[int, tuple[str, bool]] = {}
    for ann in sorted(annotations, key=lambda a: a.start):
        short = LABEL_MAP.get(ann.label, ann.label.upper())
        for i in range(ann.start, ann.end):
            char_label[i] = (short, i == ann.start)

    tokens: list[tuple[str, str]] = []
    cursor = 0
    for token in text.split():
        # Find where this token sits in the original string
        start = text.index(token, cursor)
        end = start + len(token)
        cursor = end

        # Majority-vote: take the label of the first annotated char in the token
        tag = "O"
        for i in range(start, end):
            if i in char_label:
                label, is_begin = char_label[i]
                tag = f"B-{label}" if is_begin else f"I-{label}"
                break

        tokens.append((token, tag))

    return tokens

In [17]:
texts_latin =  ['Quousque tandem abutere, Catilina patientia nostra?']

annotated_docs_latin = model.predict(texts_latin)

for text, doc in zip(texts_latin, annotated_docs_latin):
    print(f"\nText : {text}")
    print(f"Raw annotations : {doc.annotations}")

    bio = spans_to_bio(text, doc.annotations)
    print("BIO tags:")
    for token, tag in bio:
        print(f"  {token:<20} {tag}")

    # Compact list form (matches the format from your prompt)
    print("Compact:", [tag for _, tag in bio])

100%|██████████| 1/1 [00:12<00:00, 12.06s/ example]


Text : Quousque tandem abutere, Catilina patientia nostra?
Raw annotations : {Annotation(start=25, end=33, label='person', text='Catilina')}
BIO tags:
  Quousque             O
  tandem               O
  abutere,             O
  Catilina             B-PERS
  patientia            O
  nostra?              O
Compact: ['O', 'O', 'O', 'B-PERS', 'O', 'O']
